# Kalibrasi Skenario Hybrid NLP Tanpa LLM
Notebook ini menjadi alur presentasi. Logic utama tetap berada di folder `services/` agar mudah dipindahkan ke `main.py`.


In [ ]:
from pathlib import Path
import sys

import polars as pl


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.ambiguity_service import AmbiguityService
from services.artifact_service import ArtifactService
from services.clustering_service import ClusteringService
from services.dataset_service import DatasetService
from services.embedding_service import EmbeddingService
from services.evaluation_service import EvaluationService
from services.lexicon_sentiment_service import LexiconSentimentService
from services.preprocessing_service import PreprocessingService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(10_000)
pl.Config.set_tbl_width_chars(10_000)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)


## 1. Konfigurasi Dataset


In [ ]:
DATASET_PATH = config.DATASETS / "tokopedia-product-reviews-2019.csv"
OUTPUT_DATASET_PATH = config.CALIBRATION_DATASET_PATH
OUTPUT_SUMMARY_PATH = config.CALIBRATION_SUMMARY_PATH
OUTPUT_RESULTS_PATH = config.NON_LLM_RESULTS_PATH
OUTPUT_METRICS_PATH = config.NON_LLM_METRICS_PATH

RANDOM_SEED = 42
SOURCE_NAME = "tokopedia"
MAX_ROWS = 2_000  # ubah ke None jika ingin menjalankan seluruh dataset

print(f"Dataset: {DATASET_PATH}")
print(f"Batas baris demo: {MAX_ROWS if MAX_ROWS else 'semua'}")


## 2. Load, Adaptasi Kolom, Validasi


In [ ]:
dataset_service = DatasetService()
evaluation_service = EvaluationService()
artifact_service = ArtifactService()

raw_df = dataset_service.load(DATASET_PATH)
if config.COL_TEXT not in raw_df.columns:
    raise KeyError(f"Kolom '{config.COL_TEXT}' tidak ditemukan pada dataset")
if "rating" not in raw_df.columns:
    raise KeyError("Kolom 'rating' dibutuhkan sebagai label aktual kalibrasi")

base_df = raw_df
if MAX_ROWS is not None:
    base_df = base_df.head(MAX_ROWS)

text_df = (
    base_df
    .select(config.COL_TEXT, "rating")
    .with_row_index(config.COL_ID, offset=1)
    .with_columns(
        pl.col(config.COL_ID).cast(pl.Utf8),
        pl.lit(SOURCE_NAME).alias(config.COL_SOURCE),
        pl.col(config.COL_TEXT).cast(pl.Utf8, strict=False),
    )
    .select(config.COL_ID, config.COL_SOURCE, config.COL_TEXT, "rating")
)
text_df = evaluation_service.add_actual_label_from_rating(text_df)

validation = dataset_service.validate(text_df)
summary = dataset_service.build_summary(text_df)

print(f"Baris sumber: {raw_df.height:,}")
print(f"Baris digunakan: {text_df.height:,}")
print(validation)
summary


## 3. Cleaning, Deduplikasi, dan Preprocessing


In [ ]:
preprocessing_service = PreprocessingService()

clean_df = text_df.filter(
    pl.col(config.COL_TEXT).is_not_null()
    & (pl.col(config.COL_TEXT).str.strip_chars().str.len_chars() > 0)
)
clean_df = dataset_service.deduplicate(clean_df)
processed_df = preprocessing_service.process_dataframe(clean_df)
processed_df = processed_df.filter(
    pl.col(config.COL_PROCESSED).is_not_null()
    & (pl.col(config.COL_PROCESSED).str.strip_chars().str.len_chars() > 0)
)

processed_summary = dataset_service.build_summary(processed_df)
artifact_service.save_dataframe(processed_df, OUTPUT_DATASET_PATH)
dataset_service.export_summary(processed_summary, OUTPUT_SUMMARY_PATH)

print(f"Baris setelah preprocessing: {processed_df.height:,}")
processed_df.select(config.COL_ID, config.COL_TEXT, config.COL_PROCESSED, config.COL_ACTUAL_LABEL)


## 4. Rule-Based Sentiment


In [ ]:
lexicon_service = LexiconSentimentService()
rule_df = lexicon_service.analyze_dataframe(processed_df)

rule_df.select(
    config.COL_ID,
    config.COL_PROCESSED,
    config.COL_ACTUAL_LABEL,
    config.COL_RULE_LABEL,
    "rule_score",
    config.COL_RULE_CONFIDENCE,
    "rule_hits",
)


## 5. Embedding dan Clustering Semantic


In [ ]:
embedding_service = EmbeddingService(backend="hashing")
clustering_service = ClusteringService()

embedded_df, vectors = embedding_service.transform_dataframe(rule_df)
clustered_df = clustering_service.cluster_dataframe(embedded_df, vectors)

clustered_df.select(
    config.COL_ID,
    config.COL_RULE_LABEL,
    config.COL_CLUSTER_ID,
    "cluster_size",
    config.COL_SEMANTIC_LABEL,
    config.COL_SEMANTIC_SIMILARITY,
)


## 6. Ambiguity Detection Tanpa LLM


In [ ]:
ambiguity_service = AmbiguityService()
final_df = ambiguity_service.decide_dataframe(clustered_df)

final_df.select(
    config.COL_ID,
    config.COL_ACTUAL_LABEL,
    config.COL_RULE_LABEL,
    config.COL_SEMANTIC_LABEL,
    config.COL_FINAL_LABEL,
    config.COL_IS_AMBIGUOUS,
    "ambiguity_reason",
    "decision_source",
)


## 7. Evaluasi Skenario Tanpa LLM


In [ ]:
metrics = evaluation_service.evaluate_dataframe(final_df)
artifact_service.save_dataframe(final_df, OUTPUT_RESULTS_PATH)
artifact_service.save_json(metrics, OUTPUT_METRICS_PATH)

print(f"Hasil disimpan: {OUTPUT_RESULTS_PATH}")
print(f"Metrik disimpan: {OUTPUT_METRICS_PATH}")
metrics


## 8. Ringkasan Distribusi


In [ ]:
final_df.group_by(
    config.COL_ACTUAL_LABEL,
    config.COL_FINAL_LABEL,
    config.COL_IS_AMBIGUOUS,
).agg(pl.len().alias("count")).sort(
    config.COL_ACTUAL_LABEL,
    config.COL_FINAL_LABEL,
    config.COL_IS_AMBIGUOUS,
)
